# SI GRN Analysis — REF vs Carrier GRN Comparison

rs11867200 allele (REF/REF vs Carrier HET+ALT)  
One GRN inferred per allele group (age > 60, LPS + RPMI pooled).  
Comparison focuses on TF–CCL2 edges for 16 candidate TFs.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from scipy.stats import norm as sp_norm

BASE    = "/vol/projects/BIIM/agentic_immunology/temp/nienke/SI_grn_analysis"
RES_DIR = f"{BASE}/results"
IMG_DIR = f"{BASE}/images"

TFS_16 = ["CEBPB", "FOS", "FOSL2", "GATA2", "GATA3", "JUN", "JUND",
          "MAX", "MEF2A", "MYC", "NFIC", "PBX3", "TCF12",
          "SPI1", "SPIB", "ETV6"]
MOTIF_LOSS = {"SPI1", "SPIB", "ETV6"}
TARGET  = "CCL2"
COL_REF = "#2166ac"
COL_CAR = "#d6604d"

plt.rcParams.update({
    "font.family": "Arial", "font.size": 10,
    "axes.titlesize": 10, "axes.labelsize": 10,
    "xtick.labelsize": 10, "ytick.labelsize": 10,
})

# Load results
grn_ref  = pd.read_csv(f"{RES_DIR}/grn_ref.csv")
grn_car  = pd.read_csv(f"{RES_DIR}/grn_carrier.csv")
df_comp  = pd.read_csv(f"{RES_DIR}/grn_ccl2_comparison.csv")
gstats   = pd.read_csv(f"{RES_DIR}/grn_global_comparison.csv").set_index("metric")["value"]

n_ref = int(gstats.get("n_edges_REF", 0))
n_car = int(gstats.get("n_edges_Carrier", 0))

print(f"GRN_REF:     {len(grn_ref):,} edges | {grn_ref['source'].nunique()} TFs")
print(f"GRN_Carrier: {len(grn_car):,} edges | {grn_car['source'].nunique()} TFs")
print(f"Shared: {int(gstats['n_shared']):,}  REF-only: {int(gstats['n_REF_only']):,}  Carrier-only: {int(gstats['n_Carrier_only']):,}")
print(f"Jaccard: {gstats['jaccard']:.4f}")
print("\nCCL2 comparison (16 TFs):")
print(df_comp[["TF","rho_REF","rho_Carrier","delta_rho","pval_fdr","in_REF_GRN","in_Carrier_GRN","motif_loss_tf"]].to_string(index=False))

## 1. Global GRN overlap — stacked bar

In [ ]:
ref_only_n = int(gstats["n_REF_only"])
car_only_n = int(gstats["n_Carrier_only"])
shared_n   = int(gstats["n_shared"])

labels  = ["GRN_REF", "GRN_Carrier"]
totals  = [len(grn_ref), len(grn_car)]
shared_vals   = [shared_n, shared_n]
unique_vals   = [ref_only_n, car_only_n]

# figsize: 2 categories → barplot (3,3)
fig, ax = plt.subplots(figsize=(3.0, 3.0))
x = np.arange(2)
ax.bar(x, shared_vals, color="#999999", alpha=0.7, label="Shared edges")
ax.bar(x, unique_vals, bottom=shared_vals,
       color=[COL_REF, COL_CAR], alpha=0.85,
       label=[f"REF-only", "Carrier-only"])

for xi, tot in zip(x, totals):
    ax.text(xi, tot + 200, f"{tot:,}", ha="center", va="bottom", fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=10)
ax.set_ylabel("Number of edges", fontsize=10)
ax.set_title(f"GRN overlap  |  Jaccard={gstats['jaccard']:.3f}", fontsize=10)
ax.margins(x=0.3)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(handles=[
    Patch(facecolor="#999999", alpha=0.7,  label=f"Shared ({shared_n:,})"),
    Patch(facecolor=COL_REF,   alpha=0.85, label=f"REF-only ({ref_only_n:,})"),
    Patch(facecolor=COL_CAR,   alpha=0.85, label=f"Carrier-only ({car_only_n:,})"),
], fontsize=9, frameon=False, loc="upper left", bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.savefig(f"{IMG_DIR}/grn_global_overlap.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {IMG_DIR}/grn_global_overlap.png")

## 2. TF–CCL2 edge weight: REF vs Carrier barplot

In [ ]:
tf_order = [t for t in TFS_16 if t not in MOTIF_LOSS] + list(MOTIF_LOSS)

def get_val(col, tf):
    v = df_comp.loc[df_comp["TF"] == tf, col].values[0]
    return float(v) if not (isinstance(v, float) and np.isnan(v)) else np.nan

rho_r_vals = [get_val("rho_REF", t)     for t in tf_order]
rho_c_vals = [get_val("rho_Carrier", t) for t in tf_order]
in_r       = [bool(df_comp.loc[df_comp["TF"] == t, "in_REF_GRN"].values[0])     for t in tf_order]
in_c       = [bool(df_comp.loc[df_comp["TF"] == t, "in_Carrier_GRN"].values[0]) for t in tf_order]

# 16 TFs: figsize width 3 + (16-10)/5*0.5 = 3.5; +1.5 outside legend → 5.0
fig, ax = plt.subplots(figsize=(5.0, 3.0))
x = np.arange(len(tf_order))
w = 0.38

# Read actual sample sizes from AnnData obs count stored in gstats
# (fall back to edge counts if not present)
try:
    import anndata as ad
    n_ref_obs = ad.read_h5ad(f"{RES_DIR}/adata_ref.h5ad", backed="r").n_obs
    n_car_obs = ad.read_h5ad(f"{RES_DIR}/adata_carrier.h5ad", backed="r").n_obs
except Exception:
    n_ref_obs = "?"
    n_car_obs = "?"

bars1 = ax.bar(x - w/2, [v if not np.isnan(v) else 0 for v in rho_r_vals], w,
               label=f"GRN_REF (n={n_ref_obs})", color=COL_REF, alpha=0.85)
bars2 = ax.bar(x + w/2, [v if not np.isnan(v) else 0 for v in rho_c_vals], w,
               label=f"GRN_Carrier (n={n_car_obs})", color=COL_CAR, alpha=0.85)

for bar, present in zip(bars1, in_r):
    if not present:
        bar.set_hatch("///")
        bar.set_alpha(0.35)
for bar, present in zip(bars2, in_c):
    if not present:
        bar.set_hatch("///")
        bar.set_alpha(0.35)

for i, tf in enumerate(tf_order):
    row = df_comp[df_comp["TF"] == tf].iloc[0]
    if row["sig_fdr05"] or row["sig_fdr10"]:
        rr_ = row["rho_REF"]     if not np.isnan(row["rho_REF"])     else 0.0
        rc_ = row["rho_Carrier"] if not np.isnan(row["rho_Carrier"]) else 0.0
        ymax = max(abs(rr_), abs(rc_)) + 0.12
        sign = "**" if row["sig_fdr05"] else "*"
        ax.text(x[i], ymax, sign, ha="center", va="bottom", fontsize=9, clip_on=False)

for idx in [i for i, t in enumerate(tf_order) if t in MOTIF_LOSS]:
    ax.axvspan(idx - 0.5, idx + 0.5, alpha=0.10, color="orange", zorder=0)
ml_start = next((i for i, t in enumerate(tf_order) if t in MOTIF_LOSS), None)
if ml_start is not None:
    ax.text(ml_start - 0.4, 1.18, "motif-LOSS →", fontsize=9,
            color="darkorange", clip_on=False)

ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(tf_order, rotation=45, ha="right", fontsize=10)
ax.set_ylabel("Spearman ρ  (TF → CCL2)", fontsize=10)
ax.set_title(
    "GRN_REF vs GRN_Carrier — TF–CCL2 edge weight\n"
    "age>60, LPS+RPMI pooled  |  hatched=absent  |  * FDR<0.10  ** FDR<0.05",
    fontsize=10)
ax.set_ylim(-1.1, 1.35)
ax.margins(x=0.03)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(handles=[
    Patch(facecolor=COL_REF,  alpha=0.85, label=f"GRN_REF (n={n_ref_obs})"),
    Patch(facecolor=COL_CAR,  alpha=0.85, label=f"GRN_Carrier (n={n_car_obs})"),
    Patch(facecolor="gray",   hatch="///", alpha=0.35, label="Edge absent"),
    Patch(facecolor="orange", alpha=0.25, label="Motif-LOSS TFs"),
], fontsize=9, frameon=False, loc="upper left", bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.savefig(f"{IMG_DIR}/grn_ccl2_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {IMG_DIR}/grn_ccl2_comparison.png")

## 3. REF ρ vs Carrier ρ scatter (all CCL2 edges)

In [ ]:
ccl2_ref_df = (grn_ref[grn_ref["target"] == TARGET][["source","weight"]]
               .rename(columns={"weight": "rho_REF"}).set_index("source"))
ccl2_car_df = (grn_car[grn_car["target"] == TARGET][["source","weight"]]
               .rename(columns={"weight": "rho_Carrier"}).set_index("source"))

all_tfs = sorted(set(ccl2_ref_df.index) | set(ccl2_car_df.index))
srows = []
for tf in all_tfs:
    rr = float(ccl2_ref_df.loc[tf, "rho_REF"])     if tf in ccl2_ref_df.index else np.nan
    rc = float(ccl2_car_df.loc[tf, "rho_Carrier"]) if tf in ccl2_car_df.index else np.nan
    srows.append({"TF": tf, "rho_REF": rr, "rho_Carrier": rc,
                  "is_candidate": tf in TFS_16, "motif_loss": tf in MOTIF_LOSS})
sdf = pd.DataFrame(srows).dropna()

# figsize: scatter → (3,3) + 1.5 for outside legend → (4.5, 3)
fig, ax = plt.subplots(figsize=(4.5, 3.0))
bg   = sdf[~sdf["is_candidate"]]
cand = sdf[sdf["is_candidate"] & ~sdf["motif_loss"]]
ml   = sdf[sdf["motif_loss"]]

ax.scatter(bg["rho_REF"],   bg["rho_Carrier"],   s=8,  alpha=0.3,  color="gray",
           linewidths=0, label=f"Other TFs (n={len(bg)})")
ax.scatter(cand["rho_REF"], cand["rho_Carrier"], s=40, alpha=0.85, color=COL_REF,
           linewidths=0, label="Candidate TFs (13)")
ax.scatter(ml["rho_REF"],   ml["rho_Carrier"],   s=55, alpha=0.9,  color="darkorange",
           marker="^", linewidths=0, label="Motif-LOSS TFs (3)")

# Moderate density: annotate candidate + motif-loss TFs only
for _, row in pd.concat([cand, ml]).iterrows():
    ax.annotate(row["TF"], (row["rho_REF"], row["rho_Carrier"]),
                fontsize=9, xytext=(5, 3), textcoords="offset points", clip_on=False)

vmin = min(sdf["rho_REF"].min(), sdf["rho_Carrier"].min()) - 0.05
vmax = max(sdf["rho_REF"].max(), sdf["rho_Carrier"].max()) + 0.05
ax.plot([vmin, vmax], [vmin, vmax], color="black", lw=0.8, ls="--", alpha=0.5)
ax.axhline(0, color="gray", lw=0.5, ls=":")
ax.axvline(0, color="gray", lw=0.5, ls=":")
ax.set_xlabel("Spearman ρ (TF→CCL2)  GRN_REF",    fontsize=10)
ax.set_ylabel("Spearman ρ (TF→CCL2)  GRN_Carrier", fontsize=10)
ax.set_title("TF–CCL2 edge weights: REF vs Carrier\n(FDR-sig. edges only)", fontsize=10)
ax.margins(0.05)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(frameon=False, fontsize=9, loc="upper left", bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.savefig(f"{IMG_DIR}/grn_ccl2_scatter.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {IMG_DIR}/grn_ccl2_scatter.png")

## 4. Δρ barplot (Carrier − REF)

In [ ]:
df_delta  = df_comp.dropna(subset=["delta_rho"]).copy().sort_values("delta_rho")
n_items   = len(df_delta)
fh        = 3.0 + max(0, (n_items - 10) / 5) * 0.5

fig, ax = plt.subplots(figsize=(3.0, fh))
colors  = [COL_CAR if v > 0 else COL_REF for v in df_delta["delta_rho"]]
ax.barh(range(n_items), df_delta["delta_rho"].values, color=colors, alpha=0.8)

for i, (_, row) in enumerate(df_delta.iterrows()):
    if row["sig_fdr05"] or row["sig_fdr10"]:
        sign = "**" if row["sig_fdr05"] else "*"
        xpos = row["delta_rho"] + (0.01 if row["delta_rho"] >= 0 else -0.01)
        ha   = "left" if row["delta_rho"] >= 0 else "right"
        ax.text(xpos, i, sign, ha=ha, va="center", fontsize=9,
                color="black", clip_on=False)

ytlabels = [row["TF"] + (" ◆" if row["motif_loss_tf"] else "")
            for _, row in df_delta.iterrows()]
ax.set_yticks(range(n_items))
ax.set_yticklabels(ytlabels, fontsize=10)
for tick, (_, row) in zip(ax.get_yticklabels(), df_delta.iterrows()):
    if row["motif_loss_tf"]:
        tick.set_color("darkorange")

ax.axvline(0, color="black", lw=0.8, ls="--")
ax.set_xlabel("Δρ  (Carrier − REF)  TF→CCL2", fontsize=10)
ax.set_title("Allele effect on TF–CCL2 edge weight\nage>60, LPS+RPMI  |  ◆ motif-LOSS", fontsize=10)
ax.margins(y=0.03)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig(f"{IMG_DIR}/grn_delta_rho.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {IMG_DIR}/grn_delta_rho.png")

## 5. Fisher Z-test results table

In [ ]:
display_cols = ["TF", "rho_REF", "rho_Carrier", "delta_rho",
                "pval_fisher", "pval_fdr", "sig_fdr05", "sig_fdr10",
                "in_REF_GRN", "in_Carrier_GRN", "motif_loss_tf"]
df_show = df_comp[display_cols].sort_values("pval_fisher").copy()
df_show["rho_REF"]      = df_show["rho_REF"].round(3)
df_show["rho_Carrier"]  = df_show["rho_Carrier"].round(3)
df_show["delta_rho"]    = df_show["delta_rho"].round(3)
df_show["pval_fisher"]  = df_show["pval_fisher"].apply(lambda v: f"{v:.3e}" if pd.notna(v) else "NA")
df_show["pval_fdr"]     = df_show["pval_fdr"].apply(lambda v: f"{v:.3f}" if pd.notna(v) else "NA")
print(df_show.to_string(index=False))

## 6. Edge-weight heatmap: 16 TFs × 2 GRNs

In [ ]:
import matplotlib.colors as mcolors

tf_order = [t for t in TFS_16 if t not in MOTIF_LOSS] + list(MOTIF_LOSS)
hm_data  = df_comp.set_index("TF").loc[tf_order, ["rho_REF", "rho_Carrier"]]

# Heatmap: 16 TFs × 2 groups
# 2 groups on x → base (3,3); 16 TFs on y → add (16-5)/2*0.5=2.75 → 5.75 height
fig, ax = plt.subplots(figsize=(3.0, 5.5))
cmap = plt.cm.RdBu_r
norm = mcolors.TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)

im = ax.imshow(hm_data.values, aspect="auto", cmap=cmap, norm=norm)

ax.set_xticks([0, 1])
ax.set_xticklabels(["GRN_REF", "GRN_Carrier"], rotation=45, ha="right", fontsize=10)
ax.set_yticks(range(len(tf_order)))
ax.set_yticklabels(tf_order, fontsize=10)
for i, tf in enumerate(tf_order):
    if tf in MOTIF_LOSS:
        ax.get_yticklabels()[i].set_color("darkorange")

# Mark absent edges (NaN → hatched overlay)
for row_i, tf in enumerate(tf_order):
    for col_i, col in enumerate(["rho_REF", "rho_Carrier"]):
        v = hm_data.loc[tf, col]
        if np.isnan(v):
            ax.add_patch(plt.Rectangle((col_i - 0.5, row_i - 0.5), 1, 1,
                         hatch="///", fill=False, color="gray", linewidth=0))
        else:
            row_data = df_comp[df_comp["TF"] == tf].iloc[0]
            if row_data["sig_fdr05"] or row_data["sig_fdr10"]:
                sign = "**" if row_data["sig_fdr05"] else "*"
                ax.text(col_i, row_i, sign, ha="center", va="center",
                        fontsize=9, color="black", clip_on=False)

ax.set_title("TF–CCL2 edge weight\nREF vs Carrier GRN  |  * FDR<0.10", fontsize=10)
cbar = plt.colorbar(im, ax=ax, fraction=0.08, pad=0.04)
cbar.set_label("Spearman ρ", fontsize=10)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig(f"{IMG_DIR}/grn_ccl2_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {IMG_DIR}/grn_ccl2_heatmap.png")